# TerraMind latent extraction from TerraMesh S2RGB grid

This notebook runs a metadata-first pipeline for TerraMesh: query metadata with SQL, verify grid coverage, then download zarr tiles and save latent outputs to Parquet checkpoints.

In [1]:
!pip install -q huggingface_hub datasets duckdb zarr xarray pyarrow geopandas shapely folium psutil
!pip install -q git+https://github.com/IBM/terratorch.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 58.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 51.6 MB

In [1]:
import ast
import gc
import tarfile
from pathlib import Path

import geopandas as gpd
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import xarray as xr
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from IPython.display import clear_output, display
from shapely.geometry import box
from terratorch.registry import FULL_MODEL_REGISTRY
from zarr.storage import ZipStore
from huggingface_hub import HfFileSystem

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f'Using device: {device}')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Using device: cuda


In [2]:
REPO_ID = 'ibm-esa-geospatial/TerraMesh'
DATASET_SPLIT = 'train'

# Poland-like default area, configurable
BBOX = {
    'min_lat': 49.0,
    'max_lat': 55.0,
    'min_lon': 14.0,
    'max_lon': 24.0,
}

# Grid density is configurable
GRID_ROWS = 1000
GRID_COLS = 1000

# Set to an integer for shorter runs, or None for all selected cells.
MAX_SAMPLES = None

SAVE_EVERY = 1000
RESUME_FROM_CHECKPOINTS = True
INDICES = [0, 2, 4, 6, 8, 10, 12]

ENABLE_LIVE_DISPLAY = False
LIVE_REFRESH_EVERY = 25

OUTPUT_DIR = Path('outputs/latents_parquet')
HF_CACHE_DIR = Path('outputs/hf_cache')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PARQUET = OUTPUT_DIR / 'terramesh_train_metadata.parquet'

In [3]:
input_modalities = ['S2RGB']
output_modalities = ['S2RGB']

model = FULL_MODEL_REGISTRY.build(
    'terramind_v1_tiny_generate',
    modalities=input_modalities,
    output_modalities=output_modalities,
    pretrained=True,
    standardize=True,
)
model = model.to(device)
model.eval()

def resolve_s2_embedding_key(model_obj: torch.nn.Module) -> str:
    keys = list(model_obj.sampler.model.encoder_embeddings.keys())
    if 'untok_s2rgb@224' in keys:
        return 'untok_s2rgb@224'
    if 'untok_sen2l2a@224' in keys:
        return 'untok_sen2l2a@224'
    if "untok_sen2rgb@224" in keys:
        return "untok_sen2rgb@224"
    for key in keys:
        key_l = key.lower()
        if 's2' in key_l and '@224' in key_l:
            return key
    raise KeyError(f'Could not resolve an S2 embedding key. Available keys: {keys}')

S2_EMBED_KEY = resolve_s2_embedding_key(model)
print(f'Using encoder embedding key: {S2_EMBED_KEY}')

def extract_terramind_latent(model_obj: torch.nn.Module, input_batch: torch.Tensor, indices: list[int]) -> list[torch.Tensor]:
    ordered_indices = sorted(indices)
    with torch.no_grad():
        emb_out = model_obj.sampler.model.encoder_embeddings[S2_EMBED_KEY](input_batch)
        x = emb_out['x'] if isinstance(emb_out, dict) else emb_out
        latents = []

        if 0 in ordered_indices:
            layer0 = model_obj.sampler.model.encoder_norm(x).mean(dim=1)[0].detach().cpu()
            latents.append(layer0)

        for layer_idx, block in enumerate(model_obj.sampler.model.encoder, start=1):
            x = block(x)
            if layer_idx in ordered_indices:
                latent = model_obj.sampler.model.encoder_norm(x).mean(dim=1)[0].detach().cpu()
                latents.append(latent)

    if len(latents) != len(ordered_indices):
        raise RuntimeError(
            f'Latent extraction mismatch: expected {len(ordered_indices)} layers, got {len(latents)}.'
        )
    return latents

TerraMind_Tokenizer_S2L2A.pt:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [10]:
REQUIRED_COLUMNS = [
    'tar', 'zarr', 'sample_id', 'split', 'center_lon', 'center_lat', 'crs',
    'bounds', 'geometry', 'S2_time', 'S1_time', 'cloud_cover', 'delta'
]

def parse_bounds(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, str):
        value = ast.literal_eval(value)
    if not isinstance(value, (list, tuple)) or len(value) < 4:
        raise ValueError(f'Invalid bounds value: {value}')
    return [float(value[0]), float(value[1]), float(value[2]), float(value[3])]

# --- Stage 1: metadata query from HF parquet shards ---
fs = HfFileSystem()
files = fs.glob('datasets/ibm-esa-geospatial/TerraMesh/train_metadata.parquet')
if not files:
    raise FileNotFoundError('No train_metadata parquet shards found on Hugging Face.')
hf_urls = [f'hf://{f}' for f in files]

duckdb.install_extension('httpfs')
duckdb.load_extension('httpfs')

query = f"""
SELECT
    tar,
    zarr,
    sample_id,
    split,
    center_lon,
    center_lat,
    crs,
    bounds,
    geometry,
    S2_time,
    S1_time,
    cloud_cover,
    delta
FROM read_parquet({hf_urls})
WHERE center_lon BETWEEN {BBOX['min_lon']} AND {BBOX['max_lon']}
  AND center_lat BETWEEN {BBOX['min_lat']} AND {BBOX['max_lat']}
ORDER BY center_lat DESC, center_lon ASC
"""

metadata_df = duckdb.execute(query).df()
missing_columns = [c for c in REQUIRED_COLUMNS if c not in metadata_df.columns]
if missing_columns:
    raise KeyError(f'Missing required columns: {missing_columns}')

# Keep explicit types to avoid downstream ambiguity.
metadata_df['center_lon'] = pd.to_numeric(metadata_df['center_lon'], errors='raise').astype('float64')
metadata_df['center_lat'] = pd.to_numeric(metadata_df['center_lat'], errors='raise').astype('float64')
metadata_df['crs'] = pd.to_numeric(metadata_df['crs'], errors='raise').astype('int64')
metadata_df['cloud_cover'] = pd.to_numeric(metadata_df['cloud_cover'], errors='raise').astype('float64')
metadata_df['delta'] = pd.to_numeric(metadata_df['delta'], errors='raise').astype('float64')
metadata_df['S2_time'] = pd.to_datetime(metadata_df['S2_time'], errors='raise')
metadata_df['S1_time'] = pd.to_datetime(metadata_df['S1_time'], errors='raise')
metadata_df['bounds'] = metadata_df['bounds'].apply(parse_bounds)

# --- Stage 2: deterministic dense-grid selection ---
lat_edges = np.linspace(BBOX['min_lat'], BBOX['max_lat'], GRID_ROWS + 1)
lon_edges = np.linspace(BBOX['min_lon'], BBOX['max_lon'], GRID_COLS + 1)

selected_tiles = metadata_df.copy()
selected_tiles['grid_row'] = np.clip(np.digitize(selected_tiles['center_lat'], lat_edges, right=False) - 1, 0, GRID_ROWS - 1)
selected_tiles['grid_col'] = np.clip(np.digitize(selected_tiles['center_lon'], lon_edges, right=False) - 1, 0, GRID_COLS - 1)
selected_tiles['grid_id'] = selected_tiles['grid_row'].astype(str) + '_' + selected_tiles['grid_col'].astype(str)
selected_tiles['cell_center_lat'] = lat_edges[selected_tiles['grid_row'].to_numpy()] + (lat_edges[1] - lat_edges[0]) / 2
selected_tiles['cell_center_lon'] = lon_edges[selected_tiles['grid_col'].to_numpy()] + (lon_edges[1] - lon_edges[0]) / 2
selected_tiles['dist_to_cell_center'] = (
    (selected_tiles['center_lat'] - selected_tiles['cell_center_lat']) ** 2 +
    (selected_tiles['center_lon'] - selected_tiles['cell_center_lon']) ** 2
)
selected_tiles = (
    selected_tiles
    .sort_values(['grid_id', 'dist_to_cell_center', 'center_lat', 'center_lon'], ascending=[True, True, False, True])
    .drop_duplicates(subset='grid_id', keep='first')
    .sort_values(['grid_row', 'grid_col'])
    .reset_index(drop=True)
)
if MAX_SAMPLES is not None:
    selected_tiles = selected_tiles.head(MAX_SAMPLES).copy()

# --- Stage 3: coverage verification and map visualization ---
occupancy = np.zeros((GRID_ROWS, GRID_COLS), dtype=np.uint8)
occupancy[selected_tiles['grid_row'].to_numpy(), selected_tiles['grid_col'].to_numpy()] = 1
coverage_pct = 100.0 * float(occupancy.sum()) / float(occupancy.size)

VIS_MAX_FOOTPRINTS = 5000
footprint_plot_df = selected_tiles if len(selected_tiles) <= VIS_MAX_FOOTPRINTS else selected_tiles.sample(VIS_MAX_FOOTPRINTS, random_state=42)

# Keep geospatial imports local so this cell works even after a partial kernel restart.
try:
    import geopandas as gpd
    from shapely.geometry import box
except Exception as exc:
    gpd = None
    print(f'Footprint map disabled: {exc}')

gdf_parts = []
if gpd is not None:
    for crs_value, part in footprint_plot_df.groupby('crs'):
        try:
            sample_ids = []
            polygons = []
            for sample_id, bounds in zip(part['sample_id'], part['bounds']):
                if isinstance(bounds, (list, tuple, np.ndarray)) and len(bounds) == 4:
                    sample_ids.append(sample_id)
                    polygons.append(box(*bounds))
            if not polygons:
                continue
            gdf_part = gpd.GeoDataFrame({'sample_id': sample_ids}, geometry=polygons, crs=f'EPSG:{int(crs_value)}')
            gdf_parts.append(gdf_part.to_crs('EPSG:4326'))
        except Exception as exc:
            print(f'Skipping CRS {crs_value} in footprint map due to: {exc}')

gdf_bounds = gpd.GeoDataFrame(pd.concat(gdf_parts, ignore_index=True), geometry='geometry', crs='EPSG:4326') if (gpd is not None and gdf_parts) else None
verification_report = {
    'candidate_rows': int(len(metadata_df)),
    'selected_rows': int(len(selected_tiles)),
    'grid_rows': int(GRID_ROWS),
    'grid_cols': int(GRID_COLS),
    'occupied_cells': int(occupancy.sum()),
    'total_cells': int(occupancy.size),
    'coverage_pct': float(coverage_pct),
    'unique_crs': sorted(selected_tiles['crs'].dropna().unique().tolist()),
}

print(f"Metadata rows in BBOX: {verification_report['candidate_rows']:,}")
print(f"Selected grid rows: {verification_report['selected_rows']:,}")
print(f"Grid coverage: {verification_report['coverage_pct']:.4f}%")

fig, ax = plt.subplots(1, 3, figsize=(20, 6))
ax[0].scatter(selected_tiles['center_lon'], selected_tiles['center_lat'], s=1, alpha=0.6)
ax[0].set_title('Selected tile centers (lon/lat)')
ax[0].set_xlabel('longitude')
ax[0].set_ylabel('latitude')
ax[0].grid(True, alpha=0.2)

ax[1].imshow(occupancy, cmap='Greens', interpolation='nearest')
ax[1].set_title('Grid occupancy from selected tiles')
ax[1].set_xlabel('grid_col')
ax[1].set_ylabel('grid_row')

if gdf_bounds is not None and len(gdf_bounds) > 0:
    gdf_bounds.plot(ax=ax[2], linewidth=0.2, edgecolor='royalblue', facecolor='none', alpha=0.35)
    ax[2].set_title('Coverage footprints from bounds (EPSG:4326)')
    ax[2].set_xlabel('longitude')
    ax[2].set_ylabel('latitude')
else:
    ax[2].text(0.5, 0.5, 'No footprint polygons available for map', ha='center', va='center')
    ax[2].set_axis_off()

plt.tight_layout()
plt.show()

display(selected_tiles.head(3))
selected_tiles.to_parquet('terramesh_poland.parquet', index=False)

['datasets/ibm-esa-geospatial/TerraMesh/train_metadata.parquet']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                         tar                             zarr      sample_id  \
0  majortom_shard_000001.tar  majortom_train_0000168.zarr.zip  563U_165R_1_3   
1  majortom_shard_000001.tar  majortom_train_0000292.zarr.zip  592U_116R_3_2   
2  majortom_shard_000001.tar  majortom_train_0000391.zarr.zip  580U_106R_2_0   

   split  center_lon  center_lat    crs  \
0  train   23.382287   50.577712  32634   
1  train   17.513268   53.204508  32633   
2  train   15.592045   52.176477  32633   

                                              bounds  \
0  [667347.5517858453, 5604256.048244183, 669987....   
1  [666533.1229749348, 5896649.578884071, 669173....   
2  [539164.0150247386, 5779512.113980344, 541804....   

                                            geometry             S2_time  \
0  [1, 3, 0, 0, 0, 1, 0, 0, 0, 5, 0, 0, 0, 69, 10... 2021-05-11 09:30:31   
1  [1, 3, 0, 0, 0, 1, 0, 0, 0, 5, 0, 0, 0, 243, 1... 2021-03-31 10:00:21   
2  [1, 3, 0, 0, 0, 1, 0, 0, 0, 5, 0, 0, 0, 5, 134... 

In [ ]:
TAR_EXTRACT_DIR = OUTPUT_DIR / 'tar_extract_cache'
TAR_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def _download_hf_file_with_candidates(candidates: list[str]) -> Path:
    last_error = None
    for candidate in candidates:
        try:
            local_file = hf_hub_download(
                repo_id=REPO_ID,
                filename=candidate,
                repo_type='dataset',
                cache_dir=str(HF_CACHE_DIR),
                force_download=False,
            )
            return Path(local_file)
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f'Failed to download any candidate path: {candidates}') from last_error

def extract_zarr_from_tar(tar_path: Path, tar_name: str, zarr_name: str) -> Path:
    target_dir = TAR_EXTRACT_DIR / Path(tar_name).stem
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / zarr_name
    if target_path.exists():
        return target_path

    with tarfile.open(tar_path, mode='r') as tf:
        matched_member = None
        for member in tf:
            if member.name == zarr_name or member.name.endswith(f'/{zarr_name}'):
                matched_member = member
                break
        if matched_member is None:
            raise FileNotFoundError(f'{zarr_name} not found inside tar file {tar_path}')

        source = tf.extractfile(matched_member)
        if source is None:
            raise FileNotFoundError(f'Could not extract {zarr_name} from {tar_path}')
        with open(target_path, 'wb') as dst:
            dst.write(source.read())

    return target_path

def build_tar_candidates(split: str, tar_name: str) -> list[str]:
    return [
        f"{split}/S2RGB/{tar_name}",
        f"{split}/S2L2A/{tar_name}",
        tar_name,
        f"{split}/{tar_name}",
        f"train/S2RGB/{tar_name}",
        f"train/S2L2A/{tar_name}",
        f"train/{tar_name}",
    ]

def download_unique_tar_shards(rows_df: pd.DataFrame) -> dict[tuple[str, str], Path]:
    required_tars = (
        rows_df[['split', 'tar']]
        .dropna()
        .drop_duplicates()
        .sort_values(['split', 'tar'])
        .reset_index(drop=True)
    )
    total = len(required_tars)
    print(f'Tar files required for remaining rows: {total:,}')

    tar_lookup: dict[tuple[str, str], Path] = {}
    for i, tar_row in required_tars.iterrows():
        split = str(tar_row['split'])
        tar_name = str(tar_row['tar'])
        candidates = build_tar_candidates(split, tar_name)
        tar_lookup[(split, tar_name)] = _download_hf_file_with_candidates(candidates)

        if (i + 1) % 50 == 0 or (i + 1) == total:
            print(f'Downloaded/verified tar shards: {i + 1:,}/{total:,}')
    return tar_lookup

def resolve_local_zarr_path(row: pd.Series, tar_lookup: dict[tuple[str, str], Path]) -> Path:
    split = str(row['split'])
    tar_name = str(row['tar'])
    zarr_name = str(row['zarr'])
    key = (split, tar_name)
    if key not in tar_lookup:
        fallback_key = ('train', tar_name)
        if fallback_key in tar_lookup:
            key = fallback_key
        else:
            raise KeyError(f'Tar shard not pre-downloaded for split={split}, tar={tar_name}')
    return extract_zarr_from_tar(tar_lookup[key], tar_name, zarr_name)

def load_s2rgb_tensor(local_zip_path: Path) -> tuple[torch.Tensor, np.ndarray]:
    # TerraMesh samples are xarray Datasets; S2RGB data is usually under `bands`.
    with ZipStore(str(local_zip_path), mode='r') as store:
        ds = xr.open_zarr(store, consolidated=False)
        available_vars = list(ds.data_vars)

        if 'S2RGB' in ds:
            data_var = ds['S2RGB']
        elif 'bands' in ds:
            data_var = ds['bands']
        else:
            ds.close()
            raise KeyError(
                f"Expected 'S2RGB' or 'bands' variable in {local_zip_path}, found: {available_vars}"
            )

        # Most TerraMesh modalities keep imagery in (time, band, y, x).
        if 'time' in data_var.dims:
            data_var = data_var.isel(time=0)

        # Keep only the first three channels for RGB-like inputs when more bands are present.
        if 'band' in data_var.dims and data_var.sizes.get('band', 0) >= 3:
            data_var = data_var.isel(band=slice(0, 3))
        arr = data_var.values
        ds.close()

    if arr.ndim != 3:
        raise ValueError(f'Expected 3D tensor after selection, got shape {arr.shape} from {local_zip_path}')

    if arr.shape[0] == 3:
        chw = arr
    elif arr.shape[-1] == 3:
        chw = np.moveaxis(arr, -1, 0)
    else:
        raise ValueError(f'Cannot infer RGB channel axis for array shape {arr.shape}')

    chw = chw.astype(np.float32)
    if float(chw.max()) > 1.0:
        chw = np.clip(chw / 10000.0, 0.0, 1.0)
    else:
        chw = np.clip(chw, 0.0, 1.0)

    preview = np.moveaxis(chw, 0, -1)
    img = torch.from_numpy(chw)
    img = F.interpolate(
        img.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False
    )
    return img, preview

def normalize_bounds(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, tuple):
        return list(value)
    return value

def build_output_row(row: pd.Series, latents: list[torch.Tensor], indices: list[int]) -> dict:
    record = {
        'zarr': str(row['zarr']),
        'sample_id': str(row['sample_id']),
        'split': str(row['split']),
        'center_lon': float(row['center_lon']),
        'center_lat': float(row['center_lat']),
        'crs': int(row['crs']),
        'bounds': normalize_bounds(row['bounds']),
        'geometry': row['geometry'],
        'S2_time': pd.Timestamp(row['S2_time']),
        'S1_time': pd.Timestamp(row['S1_time']),
        'cloud_cover': float(row['cloud_cover']),
        'delta': float(row['delta']),
    }
    for idx, latent in zip(sorted(indices), latents):
        record[f'latent_layer_{idx}'] = latent.float().tolist()
    return record

def render_live_progress(current_rgb: np.ndarray, occupancy: np.ndarray, processed: int) -> None:
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    ax[0].imshow(np.clip(current_rgb, 0.0, 1.0))
    ax[0].set_title(f'Current S2RGB sample #{processed}')
    ax[0].axis('off')

    ax[1].imshow(occupancy, cmap='Greens', interpolation='nearest')
    ax[1].set_title('Processed grid cells')
    ax[1].set_xlabel('grid_col')
    ax[1].set_ylabel('grid_row')

    plt.tight_layout()
    display(fig)
    plt.close(fig)

In [13]:
if 'verification_report' not in globals() or 'selected_tiles' not in globals():
    raise RuntimeError('Run metadata query + verification cell before downloading zarr files.')

checkpoint_paths = sorted(OUTPUT_DIR.glob('latents_checkpoint_*.parquet'))
already_processed = 0

if RESUME_FROM_CHECKPOINTS and checkpoint_paths:
    for path in checkpoint_paths:
        already_processed += len(pd.read_parquet(path, columns=['sample_id']))
    print(f'Resuming from {already_processed:,} already-processed rows across {len(checkpoint_paths)} checkpoint files.')
else:
    print('Starting from scratch.')

rows_to_process = selected_tiles.iloc[already_processed:].reset_index(drop=True)
print(f'Rows remaining in this run: {len(rows_to_process):,}')
tar_lookup = download_unique_tar_shards(rows_to_process) if len(rows_to_process) > 0 else {}

buffer_rows = []
processed_total = already_processed
occupancy = np.zeros((GRID_ROWS, GRID_COLS), dtype=np.uint8)
if already_processed > 0:
    prefilled = selected_tiles.iloc[:already_processed]
    occupancy[prefilled['grid_row'].to_numpy(), prefilled['grid_col'].to_numpy()] = 1

for local_i, row in rows_to_process.iterrows():
    # Zarr payloads are extracted from pre-downloaded tar shards for this run.
    local_zip = resolve_local_zarr_path(row, tar_lookup)
    image_tensor, preview = load_s2rgb_tensor(local_zip)
    image_tensor = image_tensor.to(device)
    latents = extract_terramind_latent(model, image_tensor, INDICES)

    out_row = build_output_row(row, latents, INDICES)
    buffer_rows.append(out_row)

    r = int(row['grid_row'])
    c = int(row['grid_col'])
    occupancy[r, c] = 1
    processed_total += 1

    if ENABLE_LIVE_DISPLAY and (processed_total % LIVE_REFRESH_EVERY == 0 or processed_total == 1):
        render_live_progress(preview, occupancy, processed_total)

    if processed_total % SAVE_EVERY == 0:
        checkpoint_path = OUTPUT_DIR / f'latents_checkpoint_{processed_total:09d}.parquet'
        pd.DataFrame(buffer_rows).to_parquet(checkpoint_path, index=False)
        print(f'Saved checkpoint: {checkpoint_path} ({len(buffer_rows):,} rows)')
        buffer_rows.clear()
        gc.collect()

if buffer_rows:
    checkpoint_path = OUTPUT_DIR / f'latents_checkpoint_{processed_total:09d}.parquet'
    pd.DataFrame(buffer_rows).to_parquet(checkpoint_path, index=False)
    print(f'Saved final checkpoint: {checkpoint_path} ({len(buffer_rows):,} rows)')
    buffer_rows.clear()

all_checkpoints = sorted(OUTPUT_DIR.glob('latents_checkpoint_*.parquet'))
manifest = pd.DataFrame({'checkpoint_file': [str(p) for p in all_checkpoints]})
manifest_path = OUTPUT_DIR / 'checkpoints_manifest.parquet'
manifest.to_parquet(manifest_path, index=False)

print(f'Total processed rows available: {processed_total:,}')
print(f'Checkpoint files: {len(all_checkpoints)}')
print(f'Manifest written to: {manifest_path}')

RuntimeError: Run metadata verification cell before downloading zarr files.

In [ ]:
# Optional: combine checkpoints into a single parquet file.
all_checkpoints = sorted(OUTPUT_DIR.glob('latents_checkpoint_*.parquet'))
if not all_checkpoints:
    raise FileNotFoundError('No checkpoint parquet files were found to combine.')

combined_df = pd.concat((pd.read_parquet(p) for p in all_checkpoints), ignore_index=True)
combined_path = OUTPUT_DIR / 'latents_full.parquet'
combined_df.to_parquet(combined_path, index=False)
print(f'Combined parquet saved to: {combined_path}')
print(f'Combined rows: {len(combined_df):,}')